# FT-Transformer on the six C3RS events

One FT-Transformer predicts all six recorded event categories from the structured C3RS fields. It is scored exactly like the three baselines (LogReg, RandomForest, XGBoost), so the numbers here can be compared with theirs and with `train.py`.

Everything that decides the numbers is imported from the repo, not written again here:

| What | Imported from |
|---|---|
| Input features and the leakage filter | `preprocessing_data.build_features` |
| Labels and the 70 / 15 / 15 split | `preprocessing_data.make_labels`, `make_split` |
| FT-Transformer input preparation | `train.build_datasets` |
| Loss and evaluation | `utilis.loss.ClassificationLoss`, `eval.evaluate` |
| Cutoff selection and test scoring | `train.score_on_test`, which calls `utilis.metrics.score_event` |
| Baselines | `scripts.run_baselines` |
| Results folder and summary file | `utilis.results` |

**To run:** choose *Run All*, then save the notebook. It takes about 8 minutes on the GPU; the training cell in section 3 is the long one. Section 6 only reads the files written by a separate five-seed `train.py` run.

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from rtdl_revisiting_models import FTTransformer
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader

# The shared code lives in the repo root, one folder above this notebook.
ROOT = Path.cwd().resolve()
if not (ROOT / "preprocessing_data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from preprocessing_data import LEAKAGE, NOTEBOOK_DIR, build_features, make_labels, make_split, narrative_text
from scripts.run_baselines import MODELS, evaluate as score_baseline, scale_inputs
from train import build_datasets, score_on_test
from eval import evaluate
from utilis.loss import ClassificationLoss
from utilis.results import new_results_dir, summary_markdown

# Settings. These are the defaults of train.py.
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 60
PATIENCE = 8    # stop after this many epochs without a better validation mean AP
RESULTS_ROOT = ROOT / "results"
EXPERIMENT = "tabular_only/notebook"           # this notebook's own results folder under results/
FIVE_SEED_DIR = RESULTS_ROOT / "tabular_only"  # the five-seed train.py run that section 6 reads

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "-", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found, training will be slow")

# Chart style: light surface, thin grid, text in neutral ink (color is only for the data).
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
SURFACE, GRID, AXIS = "#fcfcfb", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": AXIS, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 1, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
})

## 1. Data: features, labels and split

`build_features` keeps only the structured input fields: time of day, environment, people, train and equipment. It never returns narratives, IDs, dates, event or outcome fields, or the leakage fields that restate an event (emergency or penalty brakes applied, speed restriction, type of temporary restriction).

`make_split` gives the same 70 / 15 / 15 training, validation and test reports that the baselines and `train.py` use.

In [ ]:
raw = (NOTEBOOK_DIR / os.getenv("DATA_PATH")).resolve()
df = pd.read_csv(raw, low_memory=False)
df.columns = [" ".join(str(c).split()) for c in df.columns]

Y = make_labels(df)                              # six 0/1 columns, one per event
X = build_features(df)                           # structured inputs only, leakage fields removed
tr, va, te = make_split(Y, narrative_text(df))   # row numbers of training, validation and test
events = Y.columns.tolist()

leaked = [c for c in X.columns if any(k in c for k in LEAKAGE)]
assert not leaked, leaked
print(f"{len(df):,} reports, {X.shape[1]} input features, {len(leaked)} leakage fields among them")

# Reports that have each event, per split.
counts = pd.DataFrame({name: Y.iloc[rows].sum() for name, rows in [("training", tr), ("validation", va), ("test", te)]})
counts.loc["all reports"] = [len(tr), len(va), len(te)]
counts.rename_axis("reports with the event")

## 2. Baselines

The three baselines are trained here by the same function that `scripts/run_baselines.py` uses: one yes/no model per event, settings chosen by validation AP, cutoff picked on validation, then scored once on test. This takes about half a minute.

In [ ]:
X_base = scale_inputs(X, tr)
baseline_results = pd.concat([score_baseline(name, Y, X_base, tr, va, te) for name in MODELS], ignore_index=True)

(baseline_results.groupby("model", sort=False)[["f1", "ap"]].mean()
 .rename(columns={"f1": "Average F1", "ap": "Mean AP"}).round(3))

## 3. Train the FT-Transformer

One model with six outputs, one per event. A report can have several events, so each output is its own yes/no question, trained with binary cross-entropy.

The model uses the library's default architecture and optimizer (AdamW, learning rate 1e-4). After every epoch it is checked on validation. The epoch with the best validation mean AP is kept, and training stops once validation mean AP has not improved for 8 epochs. The test set is not used in this section.

The loop follows `train_one_seed` in `train.py` step by step with the same seed, so it should reproduce the seed-42 model of a `train.py` run. Small differences are possible because GPU arithmetic is not exactly repeatable.

In [ ]:
# Same input preparation as train.py: numbers are median-filled and standardized,
# each checkbox becomes a categorical input. Everything is fitted on training rows only.
X_ft = X.copy()
X_ft.columns = [c.lower() for c in X_ft.columns]    # column names as in the processed CSVs
parts = {name: (X_ft.iloc[rows], Y.iloc[rows]) for name, rows in [("training", tr), ("validation", va), ("test", te)]}
preprocessor, datasets, cat_cardinalities = build_datasets(parts)

n_cont_features = datasets["training"].tensors[0].shape[1]
print(f"{n_cont_features} numeric and {len(cat_cardinalities)} categorical inputs")

In [ ]:
# Fixed seed: the same starting weights and the same batch order on every run.
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

loader_options = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=device.type == "cuda")
train_loader = DataLoader(datasets["training"], shuffle=True, **loader_options)
val_loader = DataLoader(datasets["validation"], shuffle=False, **loader_options)
test_loader = DataLoader(datasets["test"], shuffle=False, **loader_options)

model = FTTransformer(
    n_cont_features=n_cont_features,
    cat_cardinalities=cat_cardinalities,
    d_out=len(events),                    # one output per event
    **FTTransformer.get_default_kwargs(),
).to(device)
criterion = ClassificationLoss()          # binary cross-entropy on each event
optimizer = model.make_default_optimizer()

history = []
best_ap, best_epoch, best_state, best_val_probabilities = -1.0, 0, None, None

for epoch in range(1, MAX_EPOCHS + 1):
    start = time.perf_counter()

    model.train()
    loss_sum = 0.0
    for x_cont, x_cat, targets in train_loader:
        x_cont, x_cat, targets = x_cont.to(device), x_cat.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x_cont, x_cat).float(), targets)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * targets.size(0)
    train_loss = loss_sum / len(datasets["training"])

    val = evaluate(model, val_loader, criterion, device, desc=f"Validation {epoch}")
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val["loss"], val_mean_ap=val["mean_ap"]))

    # Keep the weights of the epoch with the best validation mean AP.
    is_best = val["mean_ap"] > best_ap
    if is_best:
        best_ap, best_epoch = val["mean_ap"], epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_val_probabilities = val["probabilities"]

    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} | validation loss {val['loss']:.4f} | "
          f"validation mean AP {val['mean_ap']:.4f} | {time.perf_counter() - start:.0f}s"
          f"{'  <- best so far' if is_best else ''}")

    if epoch - best_epoch >= PATIENCE:
        print(f"Stopped early: no improvement for {PATIENCE} epochs.")
        break

history = pd.DataFrame(history)
print(f"Kept epoch {best_epoch}: validation mean AP {best_ap:.4f}")

### Training curve

Left: the loss on training and validation reports. Right: validation mean AP, the measure used to pick the epoch. They are separate panels because they are different measures on different scales.

In [ ]:
fig, (ax_loss, ax_ap) = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
last_epoch = int(history.epoch.iloc[-1])

ax_loss.plot(history.epoch, history.train_loss, color=BLUE, label="Training")
ax_loss.plot(history.epoch, history.val_loss, color=ORANGE, label="Validation")
for column, name in [("train_loss", "Training"), ("val_loss", "Validation")]:    # name each line at its end
    ax_loss.annotate(name, (last_epoch, history[column].iloc[-1]), xytext=(6, 0), textcoords="offset points",
                     va="center", color=INK_2, fontsize=9)
ax_loss.set_title("Loss per epoch")
ax_loss.set_ylabel("Binary cross-entropy")
ax_loss.legend(frameon=False)

ax_ap.plot(history.epoch, history.val_mean_ap, color=BLUE)
ax_ap.plot(best_epoch, best_ap, "o", color=BLUE, markersize=9, markeredgecolor=SURFACE, markeredgewidth=2)
ax_ap.annotate(f"kept: epoch {best_epoch}, {best_ap:.3f}", (best_epoch, best_ap), xytext=(0, 10),
               textcoords="offset points", ha="center", color=INK, fontsize=9)
ax_ap.set_title("Validation mean AP per epoch")
ax_ap.set_ylabel("Mean AP over the six events")
ax_ap.margins(y=0.15)

for ax in (ax_loss, ax_ap):
    ax.set_xlabel("Epoch")
    ax.set_xlim(0.5, last_epoch + max(2.5, 0.18 * last_epoch))    # room on the right for the line names
    ax.xaxis.get_major_locator().set_params(integer=True)
plt.show()

## 4. Score the test set once

For each event, the yes/no cutoff is the one with the best F1 on **validation**. That cutoff is then applied to the test set. `score_on_test` is the function `train.py` uses, and it calls the same `score_event` as the baselines.

In [ ]:
model.load_state_dict(best_state)    # back to the kept epoch
test = evaluate(model, test_loader, criterion, device, desc="Test")

Y_val, Y_test = Y.iloc[va].to_numpy(), Y.iloc[te].to_numpy()
params = json.dumps(dict(seed=SEED, best_epoch=best_epoch, batch_size=BATCH_SIZE))
ft_results = score_on_test("FTTransformer", params, events, Y_val, best_val_probabilities, Y_test, test["probabilities"])

ft_results[["label", "threshold", "precision", "recall", "f1", "ap"]].set_index("label").round(3)

### FT-Transformer next to the baselines

All numbers are test scores. The first table is the average over the six events. The second has every event, with the four models side by side under each measure.

The test set has 1,540 reports, with about 210 to 380 "yes" reports per event. Resampling a set of that size moves mean AP by about 0.014, so differences of a few hundredths between models are within the noise.

In [ ]:
all_results = pd.concat([ft_results, baseline_results], ignore_index=True)
models = ["FTTransformer", *MODELS]

headline = (all_results.groupby("model", sort=False)[["f1", "ap"]].mean().reindex(models)
            .rename(columns={"f1": "Average F1", "ap": "Mean AP"}).rename_axis(None))
headline.round(3)

In [ ]:
table = (all_results.pivot(index="label", columns="model", values=["precision", "recall", "f1", "ap"])
         .reindex(index=events).reindex(columns=models, level=1)
         .rename(columns={"precision": "Precision", "recall": "Recall", "f1": "F1", "ap": "AP"}, level=0)
         .rename_axis(index="Event", columns=[None, None]))
table.loc["Average"] = table.mean()
table.round(3)

### Confusion matrix per event

FT-Transformer on the test set, at the cutoffs picked on validation. Each row is what was recorded and each column is what the model predicted. The large number is the count of reports. The percentage and the shading show the share of that row.

In [ ]:
# One blue, light to dark: darker means a larger share of the row.
shade = LinearSegmentedColormap.from_list("shade", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
predicted = test["probabilities"] >= ft_results.threshold.to_numpy()

fig, axes = plt.subplots(2, 3, figsize=(11, 7.4), constrained_layout=True)
for j, ax in enumerate(axes.flat):
    counts_j = confusion_matrix(Y_test[:, j], predicted[:, j], labels=[0, 1])
    share = counts_j / counts_j.sum(axis=1, keepdims=True)
    ax.imshow(share, cmap=shade, vmin=0, vmax=1)
    for r in range(2):
        for c in range(2):
            ink = "#ffffff" if share[r, c] > 0.6 else INK
            ax.text(c, r - 0.07, f"{counts_j[r, c]:,}", ha="center", va="center", color=ink, fontsize=15, fontweight="semibold")
            ax.text(c, r + 0.2, f"{share[r, c]:.0%} of row", ha="center", va="center", color=ink, fontsize=9)
    ax.set_title(events[j])
    ax.set_xticks([0, 1], ["No", "Yes"])
    ax.set_yticks([0, 1], ["No", "Yes"])
    ax.set_xlabel(f"Predicted (yes at {ft_results.threshold[j]:.2f} or above)")
    ax.set_ylabel("Recorded")
    # A thin gap in the background color separates the four cells.
    ax.set_xticks([0.5], minor=True)
    ax.set_yticks([0.5], minor=True)
    ax.grid(False)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="both", length=0)
    ax.spines[:].set_visible(False)
fig.suptitle("FT-Transformer on the test set: recorded vs predicted", x=0.01, ha="left", fontsize=13, fontweight="semibold")
plt.show()

## 5. Save the results

Results go to this notebook's own folder, `results/tabular_only/notebook/`: `seed_42.csv` holds every row, and `summary.md` is the short file to share. Results are never overwritten. If the folder already has results from an earlier run, a new numbered folder is used (`notebook_2`, then `notebook_3`).

In [ ]:
out = new_results_dir(EXPERIMENT, RESULTS_ROOT)
all_results.to_csv(out / f"seed_{SEED}.csv", index=False)

summary = summary_markdown(
    f"{out.name}: FT-Transformer (seed {SEED}) and baselines, from the notebook",
    [f"FT-Transformer seed {SEED}: kept epoch {best_epoch} of {len(history)}, validation mean AP {best_ap:.4f}.",
     f"Written {time.strftime('%Y-%m-%d %H:%M')}."],
    all_results,
)
(out / "summary.md").write_text(summary, encoding="utf-8")

print(summary)
print("Saved", out)

## 6. Five-seed summary from `train.py`

A single model depends a little on its random seed. `train.py` can train several models and average their probabilities. This cell trains nothing: it only reads the files in `results/tabular_only/` that such a run wrote. The run was started from the repo root with:

```
python train.py --dataset A --seeds 42 43 44 45 46
```

It takes about 35 minutes on the GPU. Because results are never overwritten, running it again writes to a new numbered folder (`results/tabular_only_2/`); set `FIVE_SEED_DIR` in the first code cell to read that folder instead.

In [ ]:
average_path = FIVE_SEED_DIR / "average.csv"
seed_paths = sorted(FIVE_SEED_DIR.glob("seed_*.csv"))

if average_path.is_file() and seed_paths:
    average = pd.read_csv(average_path)
    seed_model = average.model.iloc[0]
    print(f"{seed_model}: test scores per event")
    display(average[["label", "threshold", "precision", "recall", "f1", "ap"]].set_index("label").round(3))

    rows = []
    for path in seed_paths:
        r = pd.read_csv(path)
        info = json.loads(r.params.iloc[0])
        rows.append(dict(seed=info["seed"], best_epoch=info["best_epoch"], average_f1=r.f1.mean(), mean_ap=r.ap.mean()))
    per_seed = pd.DataFrame(rows).set_index("seed")
    print("Each of those models scored on its own:")
    display(per_seed.round(3))
    print(f"Single models: average F1 {per_seed.average_f1.mean():.3f} +/- {per_seed.average_f1.std(ddof=0):.3f}, "
          f"mean AP {per_seed.mean_ap.mean():.3f} +/- {per_seed.mean_ap.std(ddof=0):.3f} (mean +/- SD over {len(per_seed)} seeds)")

    print("\nAll models, average over the six events (test):")
    final = pd.concat([
        pd.DataFrame({"Average F1": average.f1.mean(), "Mean AP": average.ap.mean()}, index=[seed_model]),
        headline.rename(index={"FTTransformer": f"FTTransformer (seed {SEED}, this notebook)"}),
    ])
    display(final.round(3))
else:
    print("No five-seed results found in", FIVE_SEED_DIR)
    print("Run the train.py command above, then run this cell again.")